In [65]:
import time
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [66]:
df = pd.read_csv("C:\\Users\\aeaea\\OneDrive\\سطح المكتب\\ml_features_and_labels.csv")
df.head()

,e2c_total_bytes,e4_entropy_h,e5_entropy_c,e6_time_char,e6b_flow_duration_ms,e2_client_size,e2_client_record_len,e2_server_record_len,e3_cert_parsed,e1_alg_suite_Unknown(0x11eb),...,e1b_ciphersuite_60,e2b_tls_version_SSL3.0,e2b_tls_version_TLS1.0,e2b_ciphersuite_2,e2b_ciphersuite_54,e2b_ciphersuite_60,label,split,taxonomy,ID
0,8182,4.7889,5.8077,2.45,149.827957,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,train,eval_test_network,net_high_jitter_classic_run1
1,8126,5.0000,5.8367,2.44,231.020927,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,val,eval_test_network,net_high_jitter_classic_run10
2,8272,4.7889,5.9056,2.73,292.967796,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,train,eval_test_network,net_high_jitter_classic_run100
3,8272,4.8125,5.9056,2.31,156.050205,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,test,eval_test_network,net_high_jitter_classic_run101
4,8272,4.8125,5.8367,4.45,193.972111,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,test,eval_test_network,net_high_jitter_classic_run102


In [67]:
feature_columns = [col for col in df.columns if col not in ['label', 'split', 'taxonomy', 'ID']]
X = df[feature_columns].replace({True: 1, False: 0, 'TRUE': 1, 'FALSE': 0})
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
y = df['label']

X_train = X[df['split'] == 'train']
y_train = y[df['split'] == 'train']
X_val = X[df['split'] == 'val']
y_val = y[df['split'] == 'val']
X_test = X[df['split'] == 'test']
y_test = y[df['split'] == 'test']
selector = SelectKBest(mutual_info_classif, k=15)
X_train = selector.fit_transform(X_train, y_train)
X_val = selector.transform(X_val)
X_test = selector.transform(X_test)

C:\Users\aeaea\AppData\Local\Temp\ipykernel_4520\1308702904.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = df[feature_columns].replace({True: 1, False: 0, 'TRUE': 1, 'FALSE': 0})


In [68]:
model = ExtraTreesClassifier(n_estimators=400, random_state=42, n_jobs=-1)
# with class_weight balanced the results are good yeilding 91 precision and 83 recall, without balancing the class_weight the results become slighly better, by slighly i mean in decimals.
start = time.perf_counter()

# Training
model.fit(X_train, y_train)

end = time.perf_counter()
training_time = end - start
print(f'Training time: {training_time:.2f} seconds')

Training time: 1.36 seconds


In [69]:
start = time.perf_counter()
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)
end = time.perf_counter()
testing_time = end - start

results = pd.DataFrame({
    'Split': ['Validation', 'Test'],
    'Accuracy': [accuracy_score(y_val, y_val_pred), accuracy_score(y_test, y_test_pred)],
    'Precision': [precision_score(y_val, y_val_pred, zero_division=0), precision_score(y_test, y_test_pred, zero_division=0)],
    'Recall': [recall_score(y_val, y_val_pred, zero_division=0), recall_score(y_test, y_test_pred, zero_division=0)],
    'F1 Score': [f1_score(y_val, y_val_pred, zero_division=0), f1_score(y_test, y_test_pred, zero_division=0)]
})

print(f'Testing time: {testing_time:.2f} seconds')
print('\nTest classification report:')
print(classification_report(y_test, y_test_pred, zero_division=0))
results

Testing time: 0.41 seconds

Test classification report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      7000
           1       0.92      0.83      0.87      1002

    accuracy                           0.97      8002
   macro avg       0.95      0.91      0.93      8002
weighted avg       0.97      0.97      0.97      8002



,Split,Accuracy,Precision,Recall,F1 Score
0,Validation,0.967133,0.911012,0.817365,0.861652
1,Test,0.970132,0.920617,0.833333,0.874804
